# TP 1

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import psycopg2.extras as pe
import numpy as np

In [2]:
food_parquet = '../../data/food.parquet'
parquet_file = pq.ParquetFile(food_parquet)

In [3]:
parquet_columns = ["code", "brands", "product_name", "nutriments", "categories_tags", "countries_tags", "nutriscore_grade"]
first_batch = next(parquet_file.iter_batches(5_000_000, columns=parquet_columns))
df = first_batch.to_pandas()
# df.to_parquet("../../data/food_light.parquet")

In [4]:
# df = pd.read_parquet("../../data/food.parquet")

df_france = df[df['countries_tags'].astype(str).str.lower().str.contains('france', na=False)]

df_france

,code,brands,product_name,nutriments,categories_tags,countries_tags,nutriscore_grade
0,0000101209159,Bovetti,"[{'lang': 'main', 'text': 'Véritable pâte à ta...","[{'name': 'fruits-vegetables-nuts', 'value': N...","[en:breakfasts, en:spreads, en:sweet-spreads, ...",[en:france],e
17,0000130008136,NaN,"[{'lang': 'main', 'text': 'Escalope de dinde'}...",None,"[en:meats-and-their-products, en:meats, en:pou...",[en:france],unknown
18,0000140323687,NaN,"[{'lang': 'main', 'text': 'Madeleine Framboise...",None,"[en:snacks, en:sweet-snacks, en:biscuits-and-c...",[en:france],unknown
19,0000141013129,,"[{'lang': 'main', 'text': 'Croissants margarin...",None,"[en:snacks, en:sweet-snacks, en:sweet-pastries...",[en:france],unknown
24,0000171812457,NaN,"[{'lang': 'main', 'text': 'Glaces vegetales de...","[{'name': 'energy-kcal', 'value': None, '100g'...",None,[en:france],unknown
...,...,...,...,...,...,...,...
4636003,3760340550999,NaN,"[{'lang': 'main', 'text': 'Biere Sans Alcool 7...",None,[],[en:france],unknown
4636034,7707370942246,NaN,"[{'lang': 'main', 'text': 'Agua Potable Tratad...","[{'name': 'saturated-fat', 'value': None, '100...",None,[en:france],unknown
4636054,8720877258428,NaN,"[{'lang': 'main', 'text': 'Spearmint'}, {'lang...",None,None,[en:france],unknown
4636059,00276003,NaN,"[{'lang': 'main', 'text': 'Pure Whey Protein'}...",None,None,[en:france],unknown


In [5]:
# Combien de produits sont vendus en France ?
print(f"Nombre de produits vendus en Monde : {len(df)}")
print(f"Nombre de produits vendus en France : {len(df_france)}")
set_nutriscore = []
for nutriscore in df_france['nutriscore_grade'] :
    set_nutriscore.append(nutriscore)
print(set(set_nutriscore))

Nombre de produits vendus en Monde : 4636471
Nombre de produits vendus en France : 1247347
{'unknown', nan, 'c', 'a', 'd', 'b', 'not-applicable', 'e'}


In [6]:
# Quelle part du catalogue possède un Nutri-Score renseigné ?
nutriscore_grade = ['a', 'b', 'c', 'd', 'e']
count = 0
for nutriscore in df_france['nutriscore_grade']:
    if nutriscore in nutriscore_grade :
        count += 1
part_nutriscore = round((100 * count)/len(df_france), 2)
print(f"Part des Nutriscore renseigné : {part_nutriscore}%")

Part des Nutriscore renseigné : 37.18%


In [7]:
# Quelles sont les dix marques les plus présentes ?
brands = {}
for brand in df_france['brands'] :
    brand_str = str(brand)
    if brand_str.lower() not in brands:
        brands[brand_str.lower()] = 1
    else:
        brands[brand_str.lower()] += 1 

brands_sorted = dict(sorted(brands.items(), key=lambda item: item[1], reverse=True))
top_10_brands = dict(list(brands_sorted.items())[:10])
print(top_10_brands)

{'nan': 542920, '': 54688, 'carrefour': 12183, 'u': 12058, 'auchan': 6334, 'leader price': 5427, 'casino': 5251, 'le gaulois': 4550, 'cora': 4055, 'picard': 3625}


In [8]:
# Quel est le taux de valeurs manquantes sur les nutriments clés (energy_100g, sugars_100g, salt_100g) ?

nutriments = round(df_france['nutriments'].isna().mean()*100, 2)
print(f"Part des nutriments non remplis : {nutriments}%")

def is_missing(list_nutriments, name_nutriment):
    # Si la case entière est vide (NaN, None, float au lieu d'une liste...)
    if not isinstance(list_nutriments, (list, np.ndarray)):
        return True
        
    # On parcourt chaque dictionnaire de la liste
    for nutriment in list_nutriments:
        # Si on trouve le bon nutriment (ex: 'energy')
        if isinstance(nutriment, dict) and nutriment.get('name') == name_nutriment:
            valeur = nutriment.get('100g')
            # On vérifie si sa valeur est vide (None, NaN, ou chaîne vide)
            if valeur is None or pd.isna(valeur) or valeur == '':
                return True
            else:
                return False # On a trouvé une valeur valide !
                
    # Si on a fini la liste et qu'on n'a pas trouvé le nutriment, il est manquant
    return True

total_lines = len(df_france)

# On applique la fonction pour chaque nutriment ciblé
# (le .sum() va additionner tous les True, donc compter les manquants)
energy_missing = df_france['nutriments'].apply(lambda x: is_missing(x, 'energy')).sum()
sugars_missing = df_france['nutriments'].apply(lambda x: is_missing(x, 'sugars')).sum()
salt_missing = df_france['nutriments'].apply(lambda x: is_missing(x, 'salt')).sum()

# On calcule le taux (en pourcentage)
taux_energy = (energy_missing / total_lines) * 100
taux_sugars = (sugars_missing / total_lines) * 100
taux_salt = (salt_missing / total_lines) * 100

# Affichage propre des résultats
print("Taux de valeurs manquantes :")
print(f"- Énergie : {taux_energy:.2f} % ({energy_missing} manquants sur {total_lines})")
print(f"- Sucres  : {taux_sugars:.2f} % ({sugars_missing} manquants sur {total_lines})")
print(f"- Sel     : {taux_salt:.2f} % ({salt_missing} manquants sur {total_lines})")

Part des nutriments non remplis : 24.83%
Taux de valeurs manquantes :
- Énergie : 28.79 % (359124 manquants sur 1247347)
- Sucres  : 29.39 % (366555 manquants sur 1247347)
- Sel     : 33.74 % (420871 manquants sur 1247347)


In [9]:
# Selon toi, qu'est-ce qui semble le plus « sale » ou atypique dans ces données ?
# La colonne 'brands', car ses données sont soit NaN, soit vide, soit se répète, un léger changement d'orthographe qui par conséquent se considère comme unique. 
print(df.columns)

Index(['code', 'brands', 'product_name', 'nutriments', 'categories_tags',
       'countries_tags', 'nutriscore_grade'],
      dtype='str')


# TP 2

In [10]:
# Vérifier rapidement si la colonne de codes-barres contient des doublons 
print(df['code'].duplicated().value_counts()) #Nous sommes toujours sur le _light.parquet

# Filtrer DataFrame pour isoler les lignes où les valeurs de sucres sont physiquement impossibles

def verif_value_100g(list_nutriments):
    if not isinstance(list_nutriments, (list, np.ndarray)):
        return False
        
    for nutriment in list_nutriments:
        if isinstance(nutriment, dict) and nutriment.get('name') in ['sugars','energy']:
            valeur = nutriment.get('100g')
            if valeur is not None and not pd.isna(valeur) and valeur != '':
                try : 
                    valeur_num = float(valeur)
                    if nutriment.get('name') == 'energy':
                        if valeur_num == 0:
                            return True
                    elif nutriment.get('name') == 'sugars':
                        if valeur_num < 0 or valeur_num > 100:
                            return True
                except(ValueError, TypeError):
                    return False
            else:
                return False
                
    return False

df_filtre = df[df['nutriments'].apply(lambda x : verif_value_100g(x))]
print(len(df))
print(len(df_filtre))
print(f"Nombre de lignes qui n'ont aucun problème apparent : {len(df) - len(df_filtre)}")

code
False    4636411
True          60
Name: count, dtype: int64
4636471
75650
Nombre de lignes qui n'ont aucun problème apparent : 4560821


In [11]:
# Rayons couverts :

# - product_name,

# - code,

# - nutriments,

# - categories_tags

# - brands,

# - images,

# - nutriscore_grade

# - nutriscore_score

# Seuil de complétude : Le produit est conservé SI ET SEULEMENT SI les colonnes code, product_name, et l'énergie dans nutriments sont présentes à 100%"

# TP 3

*voir ~CONTRIBUTING.md*

# TP 4

*voir ~src/load_data.py*

In [12]:

df_brands = df[['brands']][df['brands'].notna()].drop_duplicates()
df_brands_list = df_brands.values.tolist()
df_brands_list

[['Bovetti'],
 ["Lagg's"],
 ['Canola Harvest'],
 ["Today's Temptations"],
 [''],
 ['Milkyway'],
 ["Mcvitie's"],
 ["Sharwood's"],
 ['allfitnessfactory.de'],
 ['Tetley,  American Power Products  Inc.'],
 ['La Fournée Campanière'],
 ['Wise Woodworks'],
 ['Intermarché'],
 ["Goode's Bakery & Co.  Inc."],
 ['Mt. Olive'],
 ["Walton's Flies"],
 ['Eco-Dent'],
 ['Lotus Brands  Inc.'],
 ['Ritter Sport,  Alfred Ritter Gmbh & Co'],
 ['Ritter Sport'],
 ['Ritter Sport,  Alfred Ritter Gmbh & Co. Kg'],
 ['Petpro Products  Inc.'],
 ["Bart & Judy's"],
 ['Lactaid'],
 ['Chio'],
 ['Lindt,  Polo Leathergoods'],
 ['Genius'],
 ['Terres et Céréales Bio'],
 ['Madelaine Chocolate Novelties'],
 ['The Madelaine Chocolate Company'],
 ['Madelaine Chocolate Novelties  Inc'],
 ['Innovative Candy Concepts'],
 ['Ocean Mist Farms'],
 ['Cean Mist Farms'],
 ['Ocean Mist'],
 ['Better Ideas'],
 ['Keepsake'],
 ['Ryan Orchards'],
 ['Sobe,  Healthy Food Brands Llc'],
 ['Healthy Food Brands Llc'],
 ['Simply,  Simply Natural Foods

## Peut être lancé indépendamment des autres

In [16]:
food_light_parquet = '../../data/food_light.parquet'
df = pd.read_parquet(food_light_parquet)

# 4. Préparation et insertion pour la table BRANDS
print("Insertion des marques...")
df_brands = df[['brands']][df['brands'].notna()].drop_duplicates()
df_brands_list = df_brands.values.tolist()
df_brands_list

Insertion des marques...


[['Bovetti'],
 ["Lagg's"],
 ['Canola Harvest'],
 ["Today's Temptations"],
 [''],
 ['Milkyway'],
 ["Mcvitie's"],
 ["Sharwood's"],
 ['allfitnessfactory.de'],
 ['Tetley,  American Power Products  Inc.'],
 ['La Fournée Campanière'],
 ['Wise Woodworks'],
 ['Intermarché'],
 ["Goode's Bakery & Co.  Inc."],
 ['Mt. Olive'],
 ["Walton's Flies"],
 ['Eco-Dent'],
 ['Lotus Brands  Inc.'],
 ['Ritter Sport,  Alfred Ritter Gmbh & Co'],
 ['Ritter Sport'],
 ['Ritter Sport,  Alfred Ritter Gmbh & Co. Kg'],
 ['Petpro Products  Inc.'],
 ["Bart & Judy's"],
 ['Lactaid'],
 ['Chio'],
 ['Lindt,  Polo Leathergoods'],
 ['Genius'],
 ['Terres et Céréales Bio'],
 ['Madelaine Chocolate Novelties'],
 ['The Madelaine Chocolate Company'],
 ['Madelaine Chocolate Novelties  Inc'],
 ['Innovative Candy Concepts'],
 ['Ocean Mist Farms'],
 ['Cean Mist Farms'],
 ['Ocean Mist'],
 ['Better Ideas'],
 ['Keepsake'],
 ['Ryan Orchards'],
 ['Sobe,  Healthy Food Brands Llc'],
 ['Healthy Food Brands Llc'],
 ['Simply,  Simply Natural Foods

In [19]:
import psycopg2
from psycopg2 import extras

def reset_database(cursor, sql_file_path):
    """Réinitialise la base de données (Idempotence)"""
    print("Réinitialisation de la base de données...")
    sql_file_path = '../../sql/postgre_creation.sql'
    with open(sql_file_path, 'r', encoding='utf-8') as file :
        sql_script = file.read()
        cursor.execute(sql_script)
    pass

conn = psycopg2.connect(
    dbname="nutriscope", 
    user="postgres", 
    password="postgres", 
    host="localhost"
)
cursor = conn.cursor()

# 2. Idempotence : on recrée les tables
reset_database(cursor, "../postgre_creation.sql")
conn.commit()

# 3. Lecture des données Pandas
print("Chargement du fichier Parquet...")
food_light_parquet = '../../data/food_light.parquet'
df = pd.read_parquet(food_light_parquet)

# 4. Préparation et insertion pour la table BRANDS
print("Insertion des marques...")
df_brands = df[['brands']][df['brands'].notna()].drop_duplicates()
df_brands_list = df_brands.values.tolist()
# - Rédige la requête INSERT appropriée
print(df_brands_list[1])
tuples_brands = [] 
query_brands = "INSERT INTO brands (name) VALUES %s ON CONFLICT (name) DO NOTHING"
extras.execute_values(cursor, query_brands, df_brands_list)

Réinitialisation de la base de données...
Chargement du fichier Parquet...
Insertion des marques...
["Lagg's"]
